# LIVR-Mini-Benchmark: GIAI ĐOẠN HUẤN LUYỆN (IMPLEMENT MINI)
Notebook này thực hiện huấn luyện mô hình **LIVR (Latent Implicit Visual Reasoning)** trên kiến trúc **Qwen2.5-VL-3B-Instruct**.

### 1. Thuật ngữ và Khái niệm cốt lõi:
- **LIVR**: Là phương pháp huấn luyện bắt buộc mô hình phải mã hóa thông tin thị giác thô thành một chuỗi các trạng thái ẩn (Latent Tokens) trước khi sử dụng chúng để suy luận hoặc trả lời câu hỏi.
- **Qwen2.5-VL-3B-Instruct**: Mô hình ngôn ngữ lớn đa phương thức (VLM) nền tảng, có khả năng hiểu ảnh, video và văn bản vượt trội.

### 2. Thiết kế Tối ưu hóa cho Tesla T4 GPU (Colab miễn phí):
Do hạ tầng Colab miễn phí giới hạn dung lượng VRAM ở mức **16GB**, chúng ta áp dụng các kỹ thuật sau:
1. **QLoRA 4-bit (NF4)**: Giảm dung lượng tải mô hình từ ~6GB xuống ~1.8GB VRAM.
2. **Float16 Compute Dtype**: Sử dụng `torch.float16` giúp tính toán nhanh trên Tensor Cores của card T4 (không dùng `bfloat16` vì T4 không hỗ trợ phần cứng này natively).
3. **Gradient Accumulation (Tích lũy Gradient)**: Chạy forward/backward với kích thước lô thực tế cực nhỏ (`batch_size_per_device = 1`) và cộng dồn gradient qua 8 bước để đạt kích thước lô hiệu dụng (`effective_batch_size = 8`).

## 1. Thiết lập Google Colab & Đồng bộ Mã nguồn
Trong phần này, chúng ta tiến hành:
1. Kết nối Google Drive để làm nơi lưu trữ checkpoint lâu dài (tránh mất mát khi phiên Colab bị ngắt kết nối).
2. Clone hoặc Pull mã nguồn mới nhất từ nhánh `develop` của kho lưu trữ GitHub cá nhân.

In [ ]:
# =========================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & ĐỒNG BỘ CODE TỪ GITHUB (DEVELOP BRANCH)
# =========================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Cấu hình URL repository của bạn
REPO_URL = "https://github.com/dinhtri445/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /content
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

##  2. Tiền Xử Lý Dữ Liệu & Lọc Trùng Lặp Ảnh Trực Quan (Visual De-duplication)
Phần này cài đặt các dependencies và chạy pipeline xử lý dữ liệu thô từ Hugging Face.
### Các khái niệm & Thuật ngữ chính:
1. **Perceptual Hashing (Mã băm nhận thức - pHash)**:
   * Khác với mã băm mật mã (như MD5/SHA256 - chỉ cần đổi 1 pixel là mã băm đổi hoàn toàn), **pHash** biểu diễn các đặc trưng cấu trúc tần số thấp của ảnh.
   * Hai ảnh tương đồng trực quan (ví dụ: cùng góc chụp nhưng lệch sáng, hoặc bị nén nhẹ) sẽ có mã băm pHash rất gần nhau (khoảng cách Hamming giữa chúng nhỏ).
   * Công thức Khoảng cách Hamming:
     $$D_H(x, y) = \sum_{i=1}^{d} (x_i \neq y_i)$$
     Nếu $D_H \le 5$, chúng ta coi như hai bức ảnh bị trùng lặp cấu trúc thị giác và loại bỏ khỏi tập Train để tránh rò rỉ tri thức (Data Leakage) sang tập Test.
2. **Dải đếm (Counting Range)**:
   * Theo tài liệu của tác giả bài báo LIVR, chỉ giữ lại các đối tượng đếm nằm trong khoảng $[2, 10]$ để tránh phân phối dữ liệu bị lệch và tối ưu hóa khả năng nhận thức của mô hình.

In [ ]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & CHẠY PIPELINE TIỀN XỬ LÝ DỮ LIỆU
# =========================================================================
# Cài đặt các thư viện cần thiết trên môi trường Colab
!pip install -r requirements.txt

import sys
import os
import torch
# Đảm bảo Python nhận diện được các module trong thư mục src/
sys.path.append(os.getcwd())

# Đường dẫn file dữ liệu sạch được lưu trữ trên Google Drive để tránh tải/lọc lại
CLEANED_DATA_PATH = "/content/drive/MyDrive/LIVR_Mini_Project/cleaned_dataset.pt"

if os.path.exists(CLEANED_DATA_PATH):
    print(f"---> Phát hiện dữ liệu sạch đã được xử lý sẵn trên Google Drive: {CLEANED_DATA_PATH}")
    print("---> Đang nạp dữ liệu sạch...")
    cleaned_dataset = torch.load(CLEANED_DATA_PATH)
    print(f"[SUCCESS] Đã nạp thành công {len(cleaned_dataset)} mẫu dữ liệu sạch trong vài giây!")
else:
    print("---> Không tìm thấy dữ liệu sạch trên Google Drive. Tiến hành tải và làm sạch từ đầu (chỉ chạy 1 lần duy nhất)...")
    from src.utils import load_and_inspect_livr_dataset, filter_and_deduplicate_pipeline
    
    # 1. Nạp và kiểm tra dữ liệu gốc (Nạp trên ổ SSD cục bộ của Colab để có tốc độ cao nhất)
    dataset = load_and_inspect_livr_dataset()
    
    # 2. Chạy bộ tiền xử lý và khử trùng lặp trực quan (Lấy mẫu cân bằng: 4 tác vụ, 300 mẫu sạch mỗi tác vụ)
    cleaned_dataset = filter_and_deduplicate_pipeline(
        dataset=dataset,
        target_tasks=['livr_counting', 'livr_object_localization', 'livr_jigsaw', 'livr_visual_similarity'],
        samples_per_task=300
    )
    
    # 3. Lưu lại kết quả đã làm sạch lên Google Drive để lần sau đọc trực tiếp
    os.makedirs(os.path.dirname(CLEANED_DATA_PATH), exist_ok=True)
    print(f"---> Đang lưu dữ liệu sạch lên Google Drive tại: {CLEANED_DATA_PATH}...")
    torch.save(cleaned_dataset, CLEANED_DATA_PATH)
    print("[SUCCESS] Đã lưu dữ liệu sạch thành công! Lần sau chạy lại sẽ không cần chờ nữa.")

## 3. Cấu Hình Kiến Trúc LIVR & Tham Số Hóa Thấp (PEFT LoRA)
Phần này khởi tạo mô hình nền tảng, nới rộng bảng từ vựng cho Latent Tokens và đóng băng có chọn lọc các tham số.
### Các khái niệm & Công thức kỹ thuật:
1. **Mở rộng bảng từ vựng (Vocab Expansion)**:
   * Chúng ta chèn thêm $K=16$ token đặc biệt (`<latent_0>` đến `<latent_15>`) vào bảng từ vựng của tokenizer và nới rộng tầng embedding của mô hình.
2. **PEFT LoRA (Low-Rank Adaptation)**:
   * Giúp huấn luyện mô hình lớn với tài nguyên cực nhỏ bằng cách đóng băng trọng số gốc $W_0 \in \mathbb{R}^{d \times k}$ và bổ sung 2 ma trận phân rã hạng thấp $A \in \mathbb{R}^{r \times k}$ và $B \in \mathbb{R}^{d \times r}$ (với hạng $r \ll \min(d, k)$, mặc định $r=16$).
   * Công thức lan truyền xuôi của trọng số LoRA:
     $$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (B \cdot A) x$$
     Trong đó $\alpha$ là hằng số tỉ lệ (scaling hyperparameter).
3. **Embedding Backward Hook (Khóa cứng biểu diễn cũ)**:
   * Chúng ta unfreeze ma trận embedding để cập nhật vector cho 16 Latent Tokens ngẫu nhiên ban đầu. Tuy nhiên, để ngăn các từ gốc bị lệch nghĩa (Token Drift), ta đăng ký một **Backward Hook** trên ma trận gradient của bảng nhúng.
   * Hook này nhân ma trận gradient $\nabla E$ với mặt nạ nhị phân $M \in \{0, 1\}^{V \times 1}$:
     $$\nabla E_{\text{hooked}} = \nabla E \odot M$$
     Trong đó $M_i = 1$ nếu $i$ thuộc danh mục index của Latent Tokens, ngược lại $M_i = 0$. Điều này triệt tiêu hoàn toàn gradient của các từ gốc và chỉ cập nhật các Latent Tokens.

In [ ]:
# =========================================================================
# CELL 3: ĐỌC CONFIG, KHỞI TẠO MÔ HÌNH VÀ BỘ TỐI ƯU
# =========================================================================
import json
import torch
from transformers import AdamW
from src.model import LIVRModelManager
from src.mask import patch_model_for_livr

# 1. Đọc file cấu hình định nghĩa sẵn
with open("config/implement_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)
print("IMPLEMENTATION CONFIGURATION:")
print(json.dumps(config, indent=2))

# 2. Dựng mô hình LIVR cấu hình K từ file config
manager = LIVRModelManager(
    model_id=config["model_id"],
    K=config["K"],
    device="cuda",
    load_in_4bit=config.get("load_in_4bit", True)
)
model = manager.setup_peft_and_freezing(
    r=config["lora_r"],
    alpha=config["lora_alpha"],
    dropout=config["lora_dropout"]
)
processor = manager.processor

# Monkey-patch model với Custom Attention Mask
patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id
)

# 3. Khai báo bộ tối ưu AdamW sử dụng cấu hình từ file config
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=config["stage1_lr"], weight_decay=config["weight_decay"])

print("-> Đã khởi tạo cấu hình model và Optimizer thành công.")

## 4. Vòng lặp Huấn luyện Hai Giai đoạn (Two-Stage Training)
### 4.1. Giao thức can thiệp mặt nạ chú ý (Two-Stage Attention Masking)
- **Stage 1 (Bottleneck Mask)**: Chặn hoàn toàn đường truyền thông tin trực tiếp từ ảnh đến Prompt và Answer. Chỉ cho phép các Latent Tokens nhìn ảnh để nén thông tin câu hỏi. Công thức Attention Mask bổ sung $A_{i, j}$ tại Layer $l$ là:
$$A_{i, j} = \begin{cases} 0 & \text{nếu } i \ge j \text{ và } \text{chặn}(i, j) = \text{False} \\ -65500.0 & \text{ngược lại} \end{cases}$$
Trong đó, $\text{chặn}(i, j) = \text{True}$ (bị chặn) khi hàng $i \in \{\text{Prompt}, \text{Answer}\}$ và cột $j \in \{\text{Image}\}$. Việc sử dụng con số phạt âm $-65500.0$ (thay cho $-\infty$) giúp tăng tính ổn định số học trong kiểu dữ liệu bán chính xác (`float16`), tránh tràn số dưới hoặc lỗi gradient biến thành `NaN` sau hàm Softmax.
- **Stage 2 (Unmasked Causal Mask)**: Gỡ bỏ mặt nạ bottleneck, cho phép mô hình nhìn ảnh trực tiếp bình thường để điều hòa tri thức ngôn ngữ.

### 4.2. Tích lũy Gradient (Gradient Accumulation)
Để tránh tràn VRAM trên Tesla T4 (dung lượng VRAM vật lý 16GB), ta sử dụng kích thước lô vật lý bằng 1 và thực hiện tích lũy gradient qua $S = 8$ bước trước khi cập nhật tham số:
$$g_{\text{accum}} = \frac{1}{S} \sum_{s=1}^{S} \nabla_{\theta} \mathcal{L}_s(\theta)$$
$$\theta \leftarrow \theta - \eta \cdot \text{AdamW}(g_{\text{accum}})$$
Trong đó $\eta$ là tốc độ học (Learning Rate) thay đổi giữa các giai đoạn ($1e-4$ ở Stage 1 và hạ xuống $5e-5$ ở Stage 2).


In [ ]:
# =========================================================================
# CELL 4: VÒNG LẶP HUẤN LUYỆN CHÍNH (TỐI ƯU HÓA VRAM & HỖ TRỢ GRADSCALER)
# =========================================================================
import os
import torch
import gc
from tqdm import tqdm
from torch.cuda.amp import GradScaler
from src.utils import prepare_vqa_inputs

# Khởi tạo bộ chia tỷ lệ gradient chống underflow cho float16
scaler = GradScaler()

# Giải phóng bộ nhớ rác trước khi huấn luyện
gc.collect()
torch.cuda.empty_cache()

# Lấy các tham số huấn luyện trực tiếp từ file config
STAGE1_EPOCHS = config["stage1_epochs"]
STAGE2_EPOCHS = config["stage2_epochs"]
TOTAL_EPOCHS = STAGE1_EPOCHS + STAGE2_EPOCHS
GRADIENT_ACCUMULATION_STEPS = config["grad_accumulation_steps"]
STAGE2_LR = config["stage2_lr"]
drive_checkpoint_dir = config["output_dir"]

model.train()
epoch_losses = []

print("====== CHÍNH THỨC KHỞI ĐỘNG VÒNG LẶP HUẤN LUYỆN 2 GIAI ĐOẠN (MEM OPTIMIZED) ======")

for epoch in range(1, TOTAL_EPOCHS + 1):
    # Xác định Giai đoạn hiện tại để điều khiển mặt nạ mã nguồn
    current_stage = 1 if epoch <= STAGE1_EPOCHS else 2
    model.livr_stage = current_stage
    
    # Reset hoặc giảm Learning Rate khi chuyển sang Stage 2 theo đúng paper
    if epoch == STAGE1_EPOCHS + 1:
        print(f"\n➔ CHUYỂN GIAI ĐOẠN: Hạ Learning Rate xuống {STAGE2_LR} cho Stage 2...")
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
            
    # Khởi tạo loss và reset gradients cho mỗi epoch ở cấp độ vòng lặp epoch
    epoch_loss = 0.0
    optimizer.zero_grad()
    
    # Thanh tiến trình theo dõi tiến độ từng epoch
    progress_bar = tqdm(cleaned_dataset, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} (Stage {current_stage})")
    
    for step, batch in enumerate(progress_bar):
        try:
            # 1. Chuẩn bị Tensor đầu vào đạt chuẩn cho Qwen (Lazy loading)
            inputs = prepare_vqa_inputs(
                processor=processor,
                conversation=batch['conversation'],
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            
            # 2. Forward pass dưới chế độ Autocast float16 để tiết kiệm VRAM
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(**inputs)
                loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
            
            # 3. Lan truyền ngược có tỉ lệ bằng GradScaler chống lỗi triệt tiêu đạo hàm
            scaler.scale(loss).backward()
            
            epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
            
            # --- THỰC THI TÍCH LŨY GRADIENT CHỐNG TRÀN VRAM ---
            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(cleaned_dataset):
                # Đưa gradients về dải thực tế trước khi cắt (clipping)
                scaler.unscale_(optimizer)
                
                # Cắt bớt gradient nếu quá lớn để chống bùng nổ đạo hàm (Gradient Clipping)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                
                # Thực hiện bước tối ưu qua scaler và cập nhật tỉ lệ
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()  # Xóa gradient
                
            # Cập nhật thông số Loss liên tục lên màn hình console
            progress_bar.set_postfix({"Loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})
            
            # 4. GIẢI PHÓNG BỘ NHỚ TRIỆT ĐỂ SAU MỖI BƯỚC
            del inputs, outputs, loss
            if step % 10 == 0:
                gc.collect()
                torch.cuda.empty_cache()
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print("\n[WARNING] Bắt gặp lỗi OOM, đang dọn cache CUDA và bỏ qua bước này...")
                optimizer.zero_grad()
                del e  # Giải phóng vết traceback tránh rò rỉ tensor
                gc.collect()
                torch.cuda.empty_cache()
                continue
            else:
                raise e
        
    avg_loss = epoch_loss / len(cleaned_dataset)
    epoch_losses.append(avg_loss)
    print(f"➔ Kết thúc Epoch {epoch} - Average Loss tổng thể: {avg_loss:.4f}")
    
    # --- LƯU CHECKPOINT TRUNG GIAN GIAI ĐOẠN 1 (STAGE 1 CHECKPOINT) ---
    if epoch == STAGE1_EPOCHS:
        stage1_checkpoint_path = os.path.join(drive_checkpoint_dir, "livr_stage1_checkpoint.pt")
        os.makedirs(drive_checkpoint_dir, exist_ok=True)
        torch.save({
            'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items() if v.requires_grad},
            'latent_embeddings': model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids].detach().cpu(),
            'latent_token_ids': manager.latent_token_ids
        }, stage1_checkpoint_path)
        print(f"---> Đã lưu checkpoint Stage 1 tại: {stage1_checkpoint_path}")

# --- LƯU CHECKPOINT CUỐI CÙNG (FINAL CHECKPOINT) ---
final_checkpoint_path = os.path.join(drive_checkpoint_dir, "livr_mini_checkpoint.pt")
os.makedirs(drive_checkpoint_dir, exist_ok=True)
torch.save({
    'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items() if v.requires_grad},
    'latent_embeddings': model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids].detach().cpu(),
    'latent_token_ids': manager.latent_token_ids
}, final_checkpoint_path)

print(f"\n[SUCCESS] Hoàn thành trọn vẹn phần Implement (Mini)!")
print(f"File trọng số thông minh đã được lưu an toàn tại: {final_checkpoint_path}")

## 5. Trực Quan Hóa Biểu Đồ Loss Nghiệm Thu
Phần này vẽ đồ thị theo dõi sự sụt giảm của hàm Loss tích lũy qua các Epoch.
*   **Mục tiêu**: Báo cáo tiến độ nghiệm thu kỹ thuật và trực quan hóa hành vi học của mô hình.
*   **Đặc điểm hình thái**: Đường Loss thường giảm mạnh ở đầu Stage 1, sau đó khi chuyển sang Stage 2 (Epoch 3) có thể dao động nhẹ do cấu trúc Attention thay đổi, nhưng sẽ nhanh chóng dốc xuống và hội tụ mượt mà ở các epoch cuối.

In [ ]:
# =========================================================================
# CELL 5: TRỰC QUAN HÓA ĐỒ THỊ LOSS
# =========================================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o', color='b', label='Training Loss')
plt.axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Transition to Stage 2')
plt.title('LIVR-Mini Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.show()

## 6. Đánh Giá Đối Chiếu Hiệu Năng (Baseline Accuracy Comparison)
Phần này chạy thử nghiệm nghiệm thu đối chiếu hiệu năng để hoàn thành báo cáo kỹ thuật.
*   **Top-1 Accuracy**: Đo lường tỉ lệ phần trăm câu trả lời sinh ra từ mô hình trùng khớp chính xác 100% với nhãn mặt đất (ground-truth targets).
*   **Adapter Disabling**: Chúng ta sử dụng hàm context manager `model.disable_adapter()` để tạm thời ngắt ma trận trọng số LoRA thích ứng, đưa mô hình quay về trạng thái gốc của hãng (Base Model) nhằm thực hiện đánh giá công bằng trên cùng tập mẫu.

In [ ]:
# =========================================================================
# CELL 6: ĐÁNH GIÁ ĐỐI CHIẾU HIỆU NĂNG BASELINE
# =========================================================================
# Chọn 20 mẫu đầu tiên của tập dữ liệu đã làm sạch để so sánh
eval_samples = cleaned_dataset[:20]

def evaluate_baseline(model, processor, manager, samples, use_lora=True):
    if use_lora:
        print("---> Đang đánh giá với LIVR LoRA Adapters (Kích hoạt)...")
        model.eval()
        model.livr_stage = 2  # Sử dụng causal mask chuẩn khi suy luận
    else:
        print("---> Đang đánh giá với Base Model gốc (Ngắt LoRA)...")
        model.eval()
        
    correct = 0
    total = 0
    
    import torch
    from src.utils import prepare_vqa_inputs
    
    with torch.no_grad():
        for item in samples:
            conv = item['conversation']
            # Chỉ gửi câu hỏi của User sang sinh output tự hồi quy
            conv_for_generation = [msg for msg in conv if msg["role"] == "user"]
            
            inputs = prepare_vqa_inputs(
                processor=processor,
                conversation=conv_for_generation,
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            inputs.pop("labels", None) # Gỡ nhãn để sinh câu trả lời tự do
            
            # Sinh chuỗi tự hồi quy
            if not use_lora:
                with model.disable_adapter():
                    outputs = model.generate(**inputs, max_new_tokens=10)
            else:
                outputs = model.generate(**inputs, max_new_tokens=10)
                
            # Lọc phần token sinh thêm ra bằng cách cắt từ độ dài input gốc để tránh trùng câu hỏi
            input_len = inputs["input_ids"].shape[1]
            pred_text = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
            target_text = str(conv[1]["content"][0]["text"]).strip()
            
            if pred_text.lower() == target_text.lower():
                correct += 1
            total += 1
            
    accuracy = (correct / total) * 100 if total > 0 else 0.0
    return accuracy

# 1. Đánh giá mô hình sau khi tinh chỉnh (LIVR LoRA)
livr_acc = evaluate_baseline(model, processor, manager, eval_samples, use_lora=True)
print(f"LIVR Model Accuracy: {livr_acc:.2f}%")

# 2. Đánh giá mô hình gốc (Base Model)
base_acc = evaluate_baseline(model, processor, manager, eval_samples, use_lora=False)
print(f"Base Model Accuracy: {base_acc:.2f}%")

print("\n" + "="*50)
print(" KẾT QUẢ ĐỐI CHIẾU HIỆU NĂNG SƠ BỘ (BASELINE COMPARISON)")
print("="*50)
print(f"Mô hình LIVR (Đã học): {livr_acc:.2f}%")
print(f"Mô hình gốc (Hãng):     {base_acc:.2f}%")
print(f"Mức độ cải thiện:        {livr_acc - base_acc:+.2f}%")
print("="*50)